# Evaluation: Static SHRED for NOAA data

Dataset: Weekly mean sea-surface temperature as given by the NOAA Optimum Interpolation SST V2 dataset [[1]](https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.html)

Model: Static SHRED from [[2]](https://royalsocietypublishing.org/rspa/article/480/2298/20240054/66770/Sensing-with-shallow-recurrent-decoder) [[3]](https://github.com/Jan-Williams/pyshred/tree/main);
Static SDN from [2][3]

In [ ]:
# Standard imports
import random

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

# Local imports
from helper_functions.evaluation import area_weighted_mean, evaluate_metrics
from helper_functions.noaa_data import crop_to_area, load_data
from helper_functions.plotting_functions import plot_compare_sst_recon
from helper_functions.saving import unpack_metadata

np.random.seed(42)
random.seed(42)

# Evaluations

## Original SHRED pipeline vs my pipeline with data leakage fixed

In [ ]:
colours = [ "#ffb000", "#dc267f", "#1589e8", "#631ff3", "#fe5100", "#000000"]  # IBM colourblind palette  green: "#0d894b"

In [ ]:
exp_name = "NOAA/SHRED-LSTM"
model_type = "SHRED"
num_sensors = 3
lags = 52
area="world"

data, lats, longs, dates = load_data()

In [ ]:
test_recons_mine = np.load(f"Reconstructions/Static/{exp_name}/recons_n{num_sensors}_l{lags}.npy")
ground_truth_mine = np.load(f"Reconstructions/Static/{exp_name}/truth_n{num_sensors}_l{lags}.npy")
sensor_locations, sensor_coords, _ = unpack_metadata(f"Reconstructions/Static/{exp_name}/metadata_n{num_sensors}_l{lags}.json")

orig_name = "NOAA/OriginalSHRED"
test_recons_original = np.load(f"Reconstructions/Static/{orig_name}/recons_n{num_sensors}_l{lags}.npy")
ground_truth_original = np.load(f"Reconstructions/Static/{orig_name}/truth_n{num_sensors}_l{lags}.npy")
sensor_locs_original = unpack_metadata(f"Reconstructions/Static/{orig_name}/metadata_n{num_sensors}_l{lags}.json")
assert np.unique(sensor_locs_original == sensor_locations)[0]

In [ ]:
lag_indices_mine = np.arange(len(test_recons_mine))
lag_indices_original = np.arange(len(test_recons_original))

In [ ]:
print("SHRED, my pipeline\nNUM SENSORS: ", num_sensors, "LAGS: ", lags)
evaluate_metrics(test_recons_mine, ground_truth_mine)

print("SHRED, original pipeline\nNUM SENSORS: ", num_sensors, "LAGS: ", lags)
evaluate_metrics(test_recons_original, ground_truth_original)

In [ ]:
difference = test_recons_mine - ground_truth_mine
plot_indexes = [3]
for i in plot_indexes:
    diff_df = pd.DataFrame({
                "Latitude": lats,
                "Longitude": longs,
                "SST": difference[i]
            })
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons_mine[i]
    })
    true_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": ground_truth_mine[i]
    })
    # plot_sst_map(diff_df, title=f"Difference map for {test_dates[i]}", min_sst=-5, max_sst=5, area="NA", cmap='bwr')
    print(f"My Pipeline Reconstruction: Lag index: {lag_indices_mine[i]}")
    plot_compare_sst_recon(recon_df, true_df, diff_data=diff_df)  #, sensor_coords=sensor_coords)

### Regional integral of SST (area weighted mean)

In [ ]:
for test_recons, ground_truth, lag_indices in [(test_recons_mine, ground_truth_mine, lag_indices_mine)]:
    print("MY PIPELINE")
    truth_sst_mean, _ = area_weighted_mean(ground_truth, lats, longs)
    test_sst_mean, _ = area_weighted_mean(test_recons, lats, longs)

    sst_mean_df = pd.DataFrame({
        "Lag index": lag_indices,
        f"SHRED (LSTM) - {num_sensors} sensors": test_sst_mean,
        "Truth": truth_sst_mean
    })
    sst_mean_df = sst_mean_df.set_index("Lag index")

    plt.figure(figsize=(8,8))
    
    sst_mean_df[[f"SHRED (LSTM) - {num_sensors} sensors",
                "Truth"]].plot(colormap=ListedColormap(colours[:2]), ls='--')
    plt.xticks(rotation=90)
    plt.ylabel("Mean SST ($\\degree$C)")
    plt.legend(bbox_to_anchor=(1,1))
    plt.show()

In [ ]:
# PROBLEM: this doesn't really show much :( consequence of shuffled test set

truth_sst_mean, _ = area_weighted_mean(ground_truth_original, lats, longs)
test_sst_mean, _ = area_weighted_mean(test_recons_original, lats, longs)

sst_mean_df = pd.DataFrame({
    f"SHRED (LSTM) - {num_sensors} sensors": test_sst_mean,
    "Truth": truth_sst_mean
})

plt.figure(figsize=(8,8))
plt.scatter(lag_indices_original, sst_mean_df[f"SHRED (LSTM) - {num_sensors} sensors"], color=colours[0], label=f"SHRED (LSTM) - {num_sensors} sensors")
plt.scatter(lag_indices_original, sst_mean_df["Truth"], color=colours[1], label="Truth")
plt.xticks(rotation=90)
plt.ylabel("Mean SST ($\\degree$C)")
plt.legend(bbox_to_anchor=(1,1))
plt.show()

### Difference
#### Area-weighted average difference

In [ ]:
difference_mine = test_recons_mine - ground_truth_mine
mean_diff_mine, _ = area_weighted_mean(difference_mine, lats, longs)
difference_original = test_recons_original - ground_truth_original
mean_diff_original, _ = area_weighted_mean(difference_original, lats, longs)

In [ ]:
for mean_diff, lag_indices, plot_type in [(mean_diff_mine, lag_indices_mine, "line"),
                                  (mean_diff_original, lag_indices_original, "scatter")]:
    sst_diff_df = pd.DataFrame({
        "Lag index": lag_indices,
        f"SHRED (LSTM) - {num_sensors} sensors": mean_diff
    })

    sst_diff_df = sst_diff_df.set_index("Lag index")

    plt.figure(figsize=(8,8))
    if plot_type == "line":
        sst_diff_df[[f"SHRED (LSTM) - {num_sensors} sensors"]].plot(colormap=ListedColormap(colours[:5]), ls='-')
    elif plot_type == "scatter": 
        plt.scatter(lag_indices_original, sst_diff_df[f"SHRED (LSTM) - {num_sensors} sensors"], color=colours[0], label=f"SHRED (LSTM) - {num_sensors} sensors")
    plt.xticks(rotation=90)
    plt.hlines(0, lag_indices[0], lag_indices[-1], colors=['k'], linestyles=['--'])
    plt.xlim(lag_indices[0], lag_indices[-1])
    plt.ylabel("Average difference (from the truth) ($\\degree$C)")
    plt.grid()
    plt.legend(bbox_to_anchor=(1,1))
    plt.show()

#### Zonally integrated average difference

In [ ]:
avg_all_time_lat_avg_diffs = []
labels = []
for (num_sensors, model_str, difference) in [(num_sensors, "SHRED (LSTM) - my pipeline", difference_mine),
                                             (num_sensors, "SHRED (LSTM) - original", difference_original)]:
    
    # For each reconstruction at a set date, find the average difference at each latitude
    latitude_avg_diffs = np.array([pd.Series(diff_at_time).groupby(lats).mean().to_numpy() for diff_at_time in difference])

    # For each latitude, calculate the mean difference over all time
    avg_all_time_lat_avg_diffs.append(np.array([np.mean(latitude_avg_diffs.T[j]) for j in range(latitude_avg_diffs.shape[1])]))
    labels.append(f"{model_str} - {num_sensors} sensors")

In [ ]:
colour_index = 0
for i in range(len(avg_all_time_lat_avg_diffs)):
    plt.plot(avg_all_time_lat_avg_diffs[i], np.unique(lats), c=colours[colour_index], label=labels[i])
    colour_index += 1
plt.vlines(0, -90, 90, colors=['k'], linestyles=['--'])
plt.ylim(-90, 90)
plt.xlabel('Average difference (from the truth) ($\\degree$C)')
plt.ylabel('Latitude')
plt.legend(bbox_to_anchor=(1,1))

## SHRED (LSTM) vs SDN - NOAA data

In [ ]:
colours = [ "#ffb000", "#dc267f", "#1589e8", "#000000"]  # "#631ff3", "#fe5100"

In [ ]:
exp_names = ["NA/SHRED-LSTM", "NA/SHRED-GRU", "NA/SDN"]
num_sensors = 3
lags = 52
area="NA"

In [ ]:
data, lats, longs, dates = load_data()
if area != "world":
    data, lats, longs = crop_to_area(data, lats, longs, area=area)

In [ ]:
test_recons = {}
ground_truth = {}
sensor_locs = {}
sensor_coords = {}
for name in exp_names:
    test_recons[name] = np.load(f"Reconstructions/Static/NOAA/{name}/recons_n{num_sensors}_l{lags}.npy")
    ground_truth[name] = np.load(f"Reconstructions/Static/NOAA/{name}/truth_n{num_sensors}_l{lags}.npy")
    locations, coords, _ = unpack_metadata(f"Reconstructions/Static/NOAA/{name}/metadata_n{num_sensors}_l{lags}.json")

    sensor_locs[name] = locations
    sensor_coords[name] = coords

In [ ]:
assert np.all(sensor_locs[exp_names[0]] == sensor_locs[exp_names[1]])
assert np.all(sensor_locs[exp_names[0]] == sensor_locs[exp_names[2]])
assert np.all(sensor_coords[exp_names[0]] == sensor_coords[exp_names[1]])
assert np.all(sensor_coords[exp_names[0]] == sensor_coords[exp_names[2]])
assert np.all(ground_truth[exp_names[0]] == ground_truth[exp_names[1]])
assert np.all(ground_truth[exp_names[0]] == ground_truth[exp_names[2]])

SENSOR_LOCS = sensor_locs[exp_names[0]]
GROUND_TRUTH = ground_truth[exp_names[0]]
SENSOR_COORDS = sensor_coords[exp_names[0]]
LAG_INDICES = np.arange(len(test_recons[exp_names[0]]))

In [ ]:
for name in exp_names:
    print(f"{name}\nNUM SENSORS: {num_sensors} LAGS: {lags}")
    evaluate_metrics(test_recons[name], ground_truth[name])

### Regional integral of SST (area weighted mean)

In [ ]:
truth_sst_mean, _ = area_weighted_mean(GROUND_TRUTH, lats, longs)
sst_mean_dict = {
    "Test index": LAG_INDICES,
    "Truth": truth_sst_mean
}
for name in exp_names:
    test_sst_mean, _ = area_weighted_mean(test_recons[name], lats, longs)
    sst_mean_dict[f"{name} - {num_sensors} sensors"] = test_sst_mean

sst_mean_df = pd.DataFrame(sst_mean_dict)
sst_mean_df = sst_mean_df.set_index("Test index")

plt.figure(figsize=(8,8))
sst_mean_df[[f"{exp_names[0]} - {num_sensors} sensors",
             f"{exp_names[1]} - {num_sensors} sensors",
             f"{exp_names[2]} - {num_sensors} sensors",
            "Truth"]].plot(colormap=ListedColormap(colours[:len(exp_names) + 1]), ls='--')
plt.xticks(rotation=90)
plt.ylabel("Mean SST ($\\degree$C)")
plt.legend(bbox_to_anchor=(1,1))
plt.grid()
plt.xlim(LAG_INDICES[0], LAG_INDICES[-1])
plt.show()

### Difference
#### Area-weighted average difference

In [ ]:
difference = {}
mean_diff = {}
for name in exp_names:
    diff = test_recons[name] - ground_truth[name]
    print(diff.shape)
    mean, _ = area_weighted_mean(diff, lats, longs)
    print(np.array(mean).shape)
    difference[name] = diff
    mean_diff[name] = np.array(mean)

In [ ]:
sst_diff_dict = {
    "Test index": LAG_INDICES,
}
for name in exp_names:
    sst_diff_dict[f"{name} - {num_sensors} sensors"] = mean_diff[name]
sst_diff_df = pd.DataFrame(sst_diff_dict)
sst_diff_df = sst_diff_df.set_index("Test index")

plt.figure(figsize=(8,8))
sst_diff_df[[f"{exp_names[0]} - {num_sensors} sensors",
             f"{exp_names[1]} - {num_sensors} sensors",
             f"{exp_names[2]} - {num_sensors} sensors",]].plot(colormap=ListedColormap(colours[:len(exp_names)]))
plt.xticks(rotation=90)
plt.hlines(0, LAG_INDICES[0], LAG_INDICES[-1], colors=['k'], linestyles=['--'])
plt.xlim(LAG_INDICES[0], LAG_INDICES[-1])
plt.ylabel("Average difference (from the truth) ($\\degree$C)")
plt.grid()
plt.legend(bbox_to_anchor=(1,1))
plt.show()

#### Zonally integrated average difference

In [ ]:
avg_all_time_lat_avg_diffs = {}
for name in exp_names:
    # For each reconstruction at a set "date", find the average difference at each latitude
    latitude_avg_diffs = np.array([pd.Series(diff_at_time).groupby(lats).mean().to_numpy() for diff_at_time in difference[name]])

    # For each latitude, calculate the mean difference over all time
    avg_all_time_lat_avg_diffs[f"{name} - {num_sensors} sensors"] = np.array([np.mean(latitude_avg_diffs.T[j]) for j in range(latitude_avg_diffs.shape[1])])

In [ ]:
colour_index = 0
for name, vals in avg_all_time_lat_avg_diffs.items():
    plt.plot(vals, np.unique(lats), c=colours[colour_index], label=name)
    colour_index += 1
plt.vlines(0, min(lats), max(lats), colors=['k'], linestyles=['--'])
if area == "world":
    plt.ylim(-90, 90)
elif area == "NA":
    plt.ylim(0, 60)
# plt.xlim(-0.8, 0.3)
plt.xlabel('Average difference (from the truth) ($\\degree$C)')
plt.ylabel('Latitude')
plt.grid()
plt.legend(bbox_to_anchor=(1,1))

In [ ]:
plot_indices = [3]
for name in exp_names:
    for i in plot_indices:
        diff_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": difference[name][i]
        })
        recon_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": test_recons[name][i]
        })
        true_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": ground_truth[name][i]
        })
        # plot_sst_map(diff_df, title=f"Difference map for {test_dates[i]}", min_sst=-5, max_sst=5, area="NA", cmap='bwr')
        print(f"{name}: Lag index: {LAG_INDICES[i]}")
        plot_compare_sst_recon(recon_df, true_df, diff_data=diff_df, area=area)  #, sensor_coords=sensor_coords)
        plot_compare_sst_recon(recon_df, true_df, diff_data=diff_df, area=area, marker_size=10)  #, sensor_coords=sensor_coords)